# Tracking Political Change with Embeddings of Parliamentary Speeches
### 2.0 Build and save the fine-tuning / probe / embedding-corpus split
Split the corpus into fine-tuning, probe, and embedding sets, using a per-speaker cap to ensure no single speaker dominates the fine-tuning data.

In [8]:
import random
import numpy as np
import pandas as pd

SEED = 24
random.seed(SEED)
np.random.seed(SEED)

In [9]:
path = "speeches_main_2000_2021_trimmed.csv"
df_full = pd.read_csv(path)
print(f"Full corpus: {len(df_full)} speeches")

MIN_SPEECHES_PER_SPEAKER = 20   # need multiple speeches per speaker to form positive pairs
MAX_SPEECHES_PER_SPEAKER = 150  # cap so frequent speakers don't dominate the subsample

speaker_counts = df_full["politicianId"].value_counts()
eligible_speakers = speaker_counts[speaker_counts >= MIN_SPEECHES_PER_SPEAKER].index
df_eligible = df_full[df_full["politicianId"].isin(eligible_speakers)].copy()

print(f"Eligible speakers (>= {MIN_SPEECHES_PER_SPEAKER} speeches): {df_eligible['politicianId'].nunique()}")
print("Party coverage among eligible speakers:")
print(df_eligible.groupby("party")["politicianId"].nunique())

# sanity check: verifies MAX_SPEECHES_PER_SPEAKER is binding for this speech corpus
print(speaker_counts.describe())
print(f"speakers over {MAX_SPEECHES_PER_SPEAKER}: {(speaker_counts > MAX_SPEECHES_PER_SPEAKER).sum()}")
print(speaker_counts[speaker_counts > MAX_SPEECHES_PER_SPEAKER])

Full corpus: 104960 speeches
Eligible speakers (>= 20 speeches): 1216
Party coverage among eligible speakers:
party
AfD              73
CDU/CSU         396
FDP             160
Fraktionslos     10
Grüne           134
LINKE           119
PDS              28
SPD             316
Name: politicianId, dtype: int64
count    1735.000000
mean       60.495677
std        70.255308
min         1.000000
25%        16.000000
50%        39.000000
75%        78.000000
max       866.000000
Name: count, dtype: float64
speakers over 150: 150
politicianId
11002625    866
11002273    633
11003255    517
11003584    492
11001171    485
           ... 
11004060    153
11004050    153
11001740    152
11003033    151
11004217    151
Name: count, Length: 150, dtype: int64


In [10]:
# sample a capped fraction of each eligible speaker's speeches into the fine-tuning set
FINETUNE_FRACTION = 0.25
MIN_FT_PER_SPEAKER = 8
MAX_FT_PER_SPEAKER = 40

def cap_and_sample_fraction(group, frac, min_n, max_n, seed):
    target = round(len(group) * frac)
    n = min(max(target, min_n), max_n, len(group))
    return group.sample(n=n, random_state=seed)

df_subsample = (
    df_eligible.groupby("politicianId", group_keys=False)
    .apply(lambda g: cap_and_sample_fraction(
        g, FINETUNE_FRACTION, MIN_FT_PER_SPEAKER, MAX_FT_PER_SPEAKER, SEED)))

print(f"Fine-tuning subsample: {len(df_subsample)} speeches "
      f"({len(df_subsample) / len(df_full):.1%} of the full corpus) | "
      f"{df_subsample['politicianId'].nunique()} speakers")
print("\n")
print("Party breakdown in fine-tuning subsample:")
print(df_subsample["party"].value_counts())

# confirm the majority of each eligible speakers speeches gets added into the embedding corpus
kept_fraction_per_speaker = 1 - (
    df_subsample.groupby("politicianId").size() / df_eligible.groupby("politicianId").size())
print("-"*30)
print("Fraction of each eligible speaker's speeches KEPT for embedding "
      "(should mostly be well above the target 75%):")
print(kept_fraction_per_speaker.describe())

finetune_speech_ids = set(df_subsample["id"])

Fine-tuning subsample: 22513 speeches (21.4% of the full corpus) | 1216 speakers


Party breakdown in fine-tuning subsample:
party
CDU/CSU         7011
SPD             5265
Grüne           3477
FDP             2901
LINKE           2649
AfD              858
PDS              264
Fraktionslos      88
Name: count, dtype: int64
------------------------------
Fraction of each eligible speaker's speeches KEPT for embedding (should mostly be well above the target 75%):
count    1216.000000
mean        0.743919
std         0.046394
min         0.600000
25%         0.742857
50%         0.750000
75%         0.754717
max         0.953811
dtype: float64


/var/folders/d2/r3c7t9m93tvb8_6r292nxt2r0000gn/T/ipykernel_32297/3571261033.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: cap_and_sample_fraction(


In [11]:
# stratified sample by party, for the held-out probe used to evaluate embedding quality
PROBE_PER_PARTY = 150

remaining_after_finetune = df_full[~df_full["id"].isin(finetune_speech_ids)].copy()

probe_df = remaining_after_finetune.groupby("party", group_keys=False).apply(
    lambda g: g.sample(n=min(len(g), PROBE_PER_PARTY), random_state=SEED))

probe_speech_ids = set(probe_df["id"])

print(f"Party-probe sample: {len(probe_df)} speeches")
print(probe_df["party"].value_counts())

Party-probe sample: 1200 speeches
party
AfD             150
CDU/CSU         150
FDP             150
Fraktionslos    150
Grüne           150
LINKE           150
PDS             150
SPD             150
Name: count, dtype: int64


/var/folders/d2/r3c7t9m93tvb8_6r292nxt2r0000gn/T/ipykernel_32297/2928179060.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  probe_df = remaining_after_finetune.groupby("party", group_keys=False).apply(


In [ ]:
embedding_corpus = df_full[~df_full["id"].isin(finetune_speech_ids)].copy() # same filter as probe_df source, reused instead of recomputed

print(f"Embedding corpus: {len(embedding_corpus)} speeches "
      f"({len(df_full) - len(embedding_corpus)} excluded as fine-tuning speeches)")

# sanity checks: fine-tuning speeches must be excluded from the embedding corpus (disjoint)
assert finetune_speech_ids.isdisjoint(set(embedding_corpus['id']))
assert probe_speech_ids.issubset(set(embedding_corpus['id']))
print("Disjointness checks passed.")

Embedding corpus: 82447 speeches (22513 excluded as fine-tuning speeches)
Disjointness checks passed.


In [13]:
# splits probe sample into val (checkpoint selection) and test (final comparison), stratified by party
PROBE_TEST_FRACTION = 0.4

def split_val_test(group, test_frac, seed):
    n_test = max(1, round(len(group) * test_frac))
    test = group.sample(n=n_test, random_state=seed)
    val = group.drop(test.index)
    return val, test

probe_val_parts, probe_test_parts = [], []
for party, group in probe_df.groupby("party"):
    val, test = split_val_test(group, PROBE_TEST_FRACTION, SEED)
    probe_val_parts.append(val)
    probe_test_parts.append(test)

probe_val_df = pd.concat(probe_val_parts).reset_index(drop=True)
probe_test_df = pd.concat(probe_test_parts).reset_index(drop=True)

probe_val_ids = set(probe_val_df["id"])
probe_test_ids = set(probe_test_df["id"])

# sanity check: val/test should partition probe_df exactly, no overlap
assert probe_val_ids.isdisjoint(probe_test_ids)
assert probe_val_ids | probe_test_ids == probe_speech_ids

print(f"Probe val:  {len(probe_val_df)} speeches")
print(probe_val_df["party"].value_counts())
print("-"*30)
print(f"Probe test: {len(probe_test_df)} speeches")
print(probe_test_df["party"].value_counts())

Probe val:  720 speeches
party
AfD             90
CDU/CSU         90
FDP             90
Fraktionslos    90
Grüne           90
LINKE           90
PDS             90
SPD             90
Name: count, dtype: int64
------------------------------
Probe test: 480 speeches
party
AfD             60
CDU/CSU         60
FDP             60
Fraktionslos    60
Grüne           60
LINKE           60
PDS             60
SPD             60
Name: count, dtype: int64


In [14]:
df_subsample.to_csv("finetune_subsample.csv", index=False)
probe_val_df.to_csv("probe_val_sample.csv", index=False)
probe_test_df.to_csv("probe_test_sample.csv", index=False)

embedding_corpus["used_in_finetune"] = False
embedding_corpus["used_in_party_probe_val"] = embedding_corpus["id"].isin(probe_val_ids)
embedding_corpus["used_in_party_probe_test"] = embedding_corpus["id"].isin(probe_test_ids)
embedding_corpus.to_csv("embedding_corpus.csv", index=False)

print("Saved:")
print(f"  finetune_subsample.csv  -- {len(df_subsample)} rows")
print(f"  probe_val_sample.csv    -- {len(probe_val_df)} rows")
print(f"  probe_test_sample.csv   -- {len(probe_test_df)} rows")
print(f"  embedding_corpus.csv    -- {len(embedding_corpus)} rows "
      f"(val probe: {embedding_corpus['used_in_party_probe_val'].sum()}, "
      f"test probe: {embedding_corpus['used_in_party_probe_test'].sum()})")

Saved:
  finetune_subsample.csv  -- 22513 rows
  probe_val_sample.csv    -- 720 rows
  probe_test_sample.csv   -- 480 rows
  embedding_corpus.csv    -- 82447 rows (val probe: 720, test probe: 480)
